# Fase 3 - Dinamica molecular con GROMACS

Partes **A** (preparacion del sistema) y **B** (simulacion) del Hito 3 del
pipeline [`pdpipe`](https://github.com/fraguero05/proyecto-proteinas).

Este notebook existe porque GROMACS no se puede instalar en el entorno de
desarrollo del proyecto: es una maquina Windows administrada sin permisos de
administrador, y GROMACS no publica binarios para Windows nativo. El plan
original del proyecto ya preveia este caso.

**El codigo es el mismo que corre localmente.** Este notebook instala el
paquete `pdpipe` y llama a las mismas funciones (`preparar_sistema`,
`simular`, `analizar`). No hay una segunda implementacion que pueda
desincronizarse de la primera.

## Antes de empezar: cuanto va a tardar

`apt install gromacs` instala una compilacion **solo para CPU**, sin CUDA.
Una GPU de Colab no la va a acelerar.

**Velocidad medida** en una corrida real sobre la ubiquitina solvatada
(26.742 atomos) en Colab gratuito con 2 nucleos: **4.0 ns/dia**. Las tres
etapas de dinamica coincidieron entre si, asi que el numero es solido.

| Cuanto simular | Cuanto tarda (2 nucleos) |
|---|---|
| 0.02 ns (validacion) | ~7 min |
| 0.2 ns | ~2.4 h |
| 2 ns (tesis) | ~12 h |

**Ojo con la equilibracion.** El NVT y el NPT son 100 ps cada uno = 100.000
pasos = ~72 min, sin importar lo corta que sea la produccion. Por eso hay un
modo de validacion rapida que tambien los acorta (ver mas abajo).

**Y bajate los resultados apenas terminen.** Colab recicla la maquina sin
aviso y se lleva todo lo que este en su disco. Si queres que sobrevivan,
usa la celda opcional de Google Drive.


## 1. Entorno


In [ ]:
# Que maquina nos toco
!python --version
!nproc --all
!free -h | head -2
!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo 'sin GPU (no importa: GROMACS de apt es solo CPU)'


In [ ]:
# GROMACS. Tarda un par de minutos.
!apt-get update -qq
!apt-get install -qq -y gromacs
!gmx --version | head -4


In [ ]:
# El pipeline, desde el repositorio. Trae los extras de datos y de MD.
!pip install -q "pdpipe[data,md] @ git+https://github.com/fraguero05/proyecto-proteinas.git"

# Si esta celda falla con 'requires a different Python', avisá: el paquete
# declara la version minima en pyproject.toml y hay que ajustarla.
import pdpipe

print('pdpipe', pdpipe.__version__, 'instalado correctamente')


In [ ]:
# OPCIONAL pero muy recomendado: trabajar sobre Google Drive.
# Colab recicla la maquina sin aviso y se lleva su disco; Drive no.
# Si lo saltas, acordate de bajar los resultados apenas terminen.
USAR_DRIVE = False

from pathlib import Path
import os

if USAR_DRIVE:
    from google.colab import drive

    drive.mount('/content/drive')
    TRABAJO = Path('/content/drive/MyDrive/pdpipe_md')
    TRABAJO.mkdir(parents=True, exist_ok=True)
    os.chdir(TRABAJO)
else:
    TRABAJO = Path('/content')

print('carpeta de trabajo:', Path.cwd())


In [ ]:
# El config.yaml del repo, para que la corrida use los mismos parametros
# que el proyecto y quede registrado en el manifiesto.
!wget -q https://raw.githubusercontent.com/fraguero05/proyecto-proteinas/main/config.yaml -O config.yaml

from pdpipe.config import load_config

cfg = load_config('config.yaml')
print('campo de fuerza:', cfg.md.force_field)
print('agua:           ', cfg.md.water_model)
print('temperatura:    ', cfg.md.temperature_k, 'K')
print('semilla:        ', cfg.seed)


## 2. Estructura de partida

Tres opciones. Descomenta la que quieras usar.


In [ ]:
from pathlib import Path

# (a) Una estructura experimental del PDB
!wget -q https://files.rcsb.org/download/1UBQ.pdb -O entrada.pdb

# (b) Un modelo de AlphaFold, por UniProt ID
# !pdpipe predict --uniprot P00698
# entrada = sorted(Path('data/processed').glob('AF-*.pdb'))[0]

# (c) Una variante disenada en la Fase 2, subida a mano
# from google.colab import files; files.upload()

entrada = Path('entrada.pdb')
print(entrada, entrada.stat().st_size, 'bytes')


## 3. Parte A - preparacion del sistema

Cinco pasos: limpieza de la estructura, topologia (`pdb2gmx`), caja
(`editconf`), solvatacion (`solvate`) e iones (`genion`).

La limpieza saca aguas cristalograficas, heteroatomos e hidrogenos
preexistentes. **Mira las advertencias**: si la estructura tenia un cofactor
o un ion catalitico, se quito, y la simulacion no lo va a tener en cuenta.


In [ ]:
from pdpipe.phase3_md import preparar_sistema

sistema = preparar_sistema(cfg, entrada, directorio='md')

print(f'\n{sistema.n_atomos:,} atomos, {sistema.n_aguas:,} aguas, iones {sistema.iones}')
print('GROMACS', sistema.version_gromacs)

if sistema.limpieza:
    L = sistema.limpieza
    print(f'limpieza: {L.atomos_iniciales} -> {L.atomos_finales} atomos, '
          f'{L.aguas_quitadas} aguas y {L.heteroatomos_quitados} heteroatomos fuera')
    for aviso in L.advertencias:
        print('AVISO:', aviso)


## 4. Parte B - las cuatro etapas

1. **Minimizacion** - saca los choques del sistema recien armado.
2. **NVT** - equilibra la temperatura, con la proteina sujeta.
3. **NPT** - equilibra la presion y la densidad del agua.
4. **Produccion** - la trayectoria que se va a analizar, sin restricciones.

Aca es donde se va el tiempo. Si Colab corta la sesion, hay que volver a
correr desde la celda de instalacion: los archivos de `md/` se pierden.


In [ ]:
# VALIDACION_RAPIDA acorta tambien la equilibracion, que si no son ~72 min
# por si sola. Sirve para comprobar que la cadena funciona, NO para sacar
# conclusiones: 10 ps no equilibran nada de verdad.
VALIDACION_RAPIDA = True

NS_PRODUCCION = 0.02 if VALIDACION_RAPIDA else 2.0

from pdpipe.phase3_md import simular

config_corrida = cfg
if VALIDACION_RAPIDA:
    config_corrida = cfg.model_copy(
        update={'md': cfg.md.model_copy(update={'nvt_ps': 10.0, 'npt_ps': 10.0})}
    )
    # Los .mdp ya estan escritos con los valores viejos: hay que rehacerlos.
    from pdpipe.phase3_md.mdp import escribir_todos

    sistema.mdp.update(
        escribir_todos(config_corrida.md, sistema.directorio, seed=cfg.seed)
    )
    print('modo validacion: NVT y NPT de 10 ps, produccion de 0.02 ns (~15 min)')

# Si la sesion se corta o cortas la celda, volve a ejecutarla: las etapas
# ya terminadas se reutilizan y la que quedo a medias se retoma desde su
# checkpoint. No se rehace lo que ya estaba bien.
resultado = simular(config_corrida, sistema, ns=NS_PRODUCCION)

print()
for etapa in resultado.etapas:
    estado = 'reutilizada' if etapa.reutilizada else f'{etapa.segundos:8.1f} s'
    print(f'{etapa.nombre:6} {etapa.pasos:>10,} pasos  {etapa.ps_simulados:>8.1f} ps  {estado}')
print(f'\ntrayectoria: {resultado.trayectoria}')
print(f'total: {resultado.duracion_total_s/60:.1f} min para {resultado.ns_simulados} ns')


## 5. Parte C - analisis

El mismo analisis que corre localmente: RMSD, RMSF, radio de giro, SASA y
puentes de hidrogeno, mas las figuras.

Aca **si** se van a contar los puentes de hidrogeno: el sistema viene de
`pdb2gmx`, asi que tiene hidrogenos explicitos y las cargas del campo de
fuerza, que es lo que MDAnalysis necesita para identificar los donantes.


In [ ]:
from pdpipe.phase3_md import analizar

# OJO con la topologia: va el .tpr, no el .gro.
# El .gro tiene solo coordenadas; el .tpr compila ademas el campo de
# fuerza con sus cargas parciales, y sin esas cargas MDAnalysis no puede
# deducir que atomos son donantes de puente de hidrogeno.
analisis = analizar(
    cfg,
    topologia=resultado.topologia_para_analisis,
    trayectoria=resultado.trayectoria,
)

for nombre, serie in analisis.series().items():
    print(f'{nombre:12} media {serie.media:>10.3f} {serie.unidad:4} '
          f'deriva {serie.deriva:>+9.3f}')

if analisis.rmsf:
    print('
residuos mas moviles:', analisis.rmsf.mas_moviles(5))
for aviso in analisis.advertencias:
    print('AVISO:', aviso)


In [ ]:
# El panel resumen
from IPython.display import Image, display

for figura in analisis.figuras:
    if figura.name.endswith('_resumen.png'):
        display(Image(str(figura)))


## 6. Bajar los resultados

**Hacelo antes de cerrar la pestana.** Todo lo que esta en el disco de Colab
se borra al terminar la sesion.

El ZIP incluye el `run_manifest.json`, que es lo que hace reproducible la
corrida: versiones, parametros, semillas y hashes de los archivos.


In [ ]:
import shutil
from google.colab import files

# La trayectoria puede pesar bastante; se incluye igual porque es el insumo
# de la Fase 5.
for carpeta in ('md', 'data', 'runs'):
    if Path(carpeta).is_dir():
        shutil.make_archive(f'resultados_{carpeta}', 'zip', carpeta)
        kb = Path(f'resultados_{carpeta}.zip').stat().st_size // 1024
        print(f'resultados_{carpeta}.zip', kb, 'KB')

for z in sorted(Path('.').glob('resultados_*.zip')):
    files.download(str(z))


---

## Seguir en la maquina local

Descomprimi el ZIP en el repo y volve a analizar la trayectoria localmente:

```powershell
pdpipe md-analyze --topology md/prod.tpr --trajectory md/prod.xtc
```

Deberia dar los mismos numeros que la celda 5. Si no coinciden, algo se
perdio en la descarga.

## Que mirar para saber si la simulacion sirvio

- **RMSD que se aplana.** Si al final sigue subiendo, la produccion fue corta
  y el sistema todavia no se equilibro. Es lo primero a revisar con 0.2 ns.
- **Radio de giro estable.** Si crece sostenidamente, la proteina se
  despliega: puede ser real o puede ser un problema de preparacion.
- **Puentes de hidrogeno estables.** Si caen, se pierde estructura secundaria.
- **RMSF con picos en extremos y loops.** Un RMSF plano suele indicar que algo
  quedo mal sujeto o mal alineado.
